# Lab: Tool Calling — From Schema to Plugin System

**Module 02 — Function Calling & Tool Systems**

## Objectives

By the end of this lab you will be able to:

1. **Extract** structured data from text using `response_format` and Pydantic
2. **Evaluate** AI-generated JSON Schemas and know what makes them good or bad
3. **Use Pydantic** to define schemas and validate structured output
4. **Implement** a tool execution function with structured error returns
5. **Wire** a tool into an LLM API using the two-call pattern with LiteLLM
6. **Build** a decorator-based `ToolRegistry` to manage multiple tools as a plugin system
7. **Measure** the latency win from running independent tool calls with `asyncio.gather`

---

## The Story

We start by forcing the model to return a simple structured object, then move to a single calculator tool, then to a production-ready plugin system, and finish by parallelising independent calls.

| Part | Topic |
|------|-------|
| 1 | Structured Output — Data Extraction with Pydantic |
| 2 | Schema Design — AI-assisted, human-reviewed |
| 3 | Pydantic — Type-safe schemas & validation |
| 4 | Tool Execution — The return contract |
| 5 | The Two-Call API Pattern |
| 6 | Live Demo |
| 7 | Python Concept: Decorators |
| 8 | Building a Plugin System with a Tool Registry |
| 9 | Live Demo with the Registry |
| 10 | Parallel Tool Calls — measuring `asyncio.gather` |


In [ ]:
# Setup
# !uv pip install litellm pydantic python-dotenv

import json
import os
import logging
import time
import inspect
from threading import Lock
from typing import Optional, List, Dict, Any, Callable
from enum import Enum
from pydantic import BaseModel, Field, ValidationError, create_model
from dotenv import load_dotenv
import litellm

load_dotenv()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
litellm.drop_params = True

---
## Part 1: Structured Output (Data Extraction)

Before we look at tools, let's look at **Structured Output**. 
This is the simplest way to get data out of an LLM in a reliable, machine-readable format.

Instead of getting a raw string and trying to parse it with regex, we pass a **Pydantic model** to the API. 
The model is guaranteed to match the schema because the LLM provider uses **Constraint Sampling** to ensure valid tokens.

For details on provider support and configuration options, see the [LiteLLM JSON Mode Docs](https://docs.litellm.ai/docs/completion/json_mode).

### Walkthrough: Sentiment Analysis with `response_format`

In [ ]:
from pydantic import BaseModel
from typing import Literal

class SentimentResult(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float

def analyze_sentiment(text: str) -> SentimentResult:
    response = litellm.completion(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": f"Analyze this sentiment: {text}"}],
        response_format=SentimentResult
    )
    # Deserialize the raw JSON string into the Pydantic object
    return SentimentResult.model_validate_json(response.choices[0].message.content)

result = analyze_sentiment("I absolutely love this new tool calling lab!")
print(f"Sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence:.2f}")

### Exercise: Urgency Check

Define a model called `UrgencyCheck` with a single boolean field `is_urgent`.
Then, use it to classify a user message.

In [ ]:
# TODO: Define the UrgencyCheck model
class UrgencyCheck(BaseModel):
    pass

# TODO: Call the API and parse the response
def check_urgency(text: str) -> UrgencyCheck:
    pass

test_text = "HELP! My production server is down and I can't reach my customers!"
# print(check_urgency(test_text))

---
## Part 2: Schema Design

A tool schema tells the LLM **what** the tool does and **how** to call it.
The `description` field is essentially a prompt — it determines *when* the model decides to use the tool.

### Walkthrough: What a good schema looks like

| Principle | Why it matters |
|-----------|----------------|
| **Verb name** (`execute_calculation`) | Helps the model understand it's an action |
| **Rich description** with examples | The model uses this to decide *when* to call the tool |
| **Enums over free text** | Constrains the model's output — fewer hallucinations |
| **All required** | No optional fields unless you explicitly mean it |

In [ ]:
# Here is a well-designed schema. Read it carefully — pay attention to the description.
calculator_schema = {
    "type": "function",
    "function": {
        "name": "execute_calculation",
        "description": (
            "Executes a basic arithmetic operation. "
            "Use for any math in user questions: percentages, growth rates, "
            "compound interest, splits, or simple arithmetic. "
            "Example: For 'What is 15% of 200?', use operation='multiply', "
            "operand_a=200, operand_b=0.15."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation to perform."
                },
                "operand_a": {
                    "type": "number",
                    "description": "The first operand."
                },
                "operand_b": {
                    "type": "number",
                    "description": "The second operand (divisor for 'divide')."
                }
            },
            "required": ["operation", "operand_a", "operand_b"]
        }
    }
}

print(json.dumps(calculator_schema, indent=2))

### Exercise: AI-Assisted Schema Design

> [!NOTE]
> In practice, you'll use AI to *generate* schemas and your job is to **review and improve** them.
> That skill — reading and evaluating a schema — is more valuable than hand-writing JSON.

**Step 1:** Ask an AI assistant (ChatGPT, Copilot, Gemini) to generate a schema for this tool:

> *"Generate an OpenAI-compatible JSON tool schema for a function called `search_hotels`.*
> *It takes: `location` (city name), `price_range` (one of budget/mid/luxury),*
> *and `amenities` (array of strings from: pool, wifi, gym, parking, restaurant).*
> *Write a rich description with an example."*

**Step 2:** Paste the AI output below and run the checker.

**Step 3:** Fix any issues the checker finds. Common AI mistakes:
- Missing `enum` on `price_range` or `amenities.items`
- Weak description (no example, no edge cases)
- Missing fields in `required`

In [ ]:
# TODO: Paste the AI-generated schema here, then review and fix it.
# The schema structure should follow the calculator_schema above.

search_hotels_schema = {
    "type": "function",
    "function": {
        "name": "search_hotels",
        "description": "TODO: Replace with AI-generated (and reviewed) description.",
        "parameters": {
            "type": "object",
            "properties": {
                # TODO: Paste parameters from the AI output here.
                # Make sure price_range has an enum and amenities items has an enum.
            },
            "required": []  # TODO: Which fields should be required?
        }
    }
}

print(json.dumps(search_hotels_schema, indent=2))

In [ ]:
# Validation check
from checker.lab01 import check_hotel_schema
check_hotel_schema(search_hotels_schema)

---
## Part 3: Pydantic Schemas

Hand-written JSON dicts don't scale. **Pydantic** gives you:

1. **Single source of truth** — define schema once in Python, generate JSON automatically
2. **Automatic validation** — catches invalid LLM output before it reaches your code
3. **Type safety** — IDE autocomplete and type checking

### Walkthrough: Calculator as a Pydantic Model

In [ ]:
class Operation(str, Enum):
    ADD = "add"
    SUBTRACT = "subtract"
    MULTIPLY = "multiply"
    DIVIDE = "divide"


class CalculationRequest(BaseModel):
    """
    Executes a basic arithmetic operation.
    Use for any math in user questions: percentages, growth rates,
    compound interest, splits, or simple arithmetic.
    Example: For 'What is 15% of 200?', use operation='multiply',
    operand_a=200, operand_b=0.15.
    """
    operation: Operation = Field(description="The arithmetic operation to perform.")
    operand_a: float = Field(description="The first operand.")
    operand_b: float = Field(description="The second operand (divisor for 'divide').")


# This replaces the hand-written dict — generated automatically from the class
CALCULATOR_SCHEMA = {
    "type": "function",
    "function": {
        "name": "execute_calculation",
        "description": CalculationRequest.__doc__.strip(),
        "parameters": CalculationRequest.model_json_schema()
    }
}

print(json.dumps(CALCULATOR_SCHEMA, indent=2))

### Exercise: Hotel Search Result Model

> [!NOTE]
> **Structured Output vs. Tool Parameters**: This Pydantic model parses the LLM's *response*.
> This is different from defining the *parameters* the LLM sends to a tool.
> Both use Pydantic, but they serve different parts of the agent loop.

Define a Pydantic model for a hotel search result:
- `name`: string
- `city`: string
- `price_per_night`: float (must be > 0)
- `rating`: float (between 1.0 and 5.0)
- `amenities`: list of strings

In [ ]:
# TODO: Define the HotelResult Pydantic model
# Hint: Use Field(gt=0) for price, Field(ge=1.0, le=5.0) for rating

class HotelResult(BaseModel):
    """Structured result for a hotel search."""
    # TODO: Define the fields with appropriate types and constraints
    pass


print(json.dumps(HotelResult.model_json_schema(), indent=2))

In [ ]:
# Validation check
from checker.lab01 import check_hotel_model
check_hotel_model(HotelResult)

---
## Part 4: Tool Execution

The tool execution function must:
- **Always** return a dict with `success`, `result`, `error`
- **Never** raise an uncaught exception — return structured errors instead
- Handle domain errors (e.g., division by zero) explicitly

### Walkthrough: The return contract

In [ ]:
# Every tool always returns one of these two shapes:
SUCCESS = {"success": True,  "result": 42.0, "error": None}
FAILURE = {"success": False, "result": None,  "error": "Division by zero is not allowed."}

# This consistent contract means the agent loop never needs to special-case tool results.

### Exercise: Implement `execute_calculation`

In [ ]:
def execute_calculation(operation: str, operand_a: float, operand_b: float) -> Dict[str, Any]:
    """
    Performs the calculation and returns a structured result.

    Args:
        operation: One of "add", "subtract", "multiply", "divide"
        operand_a: The first operand
        operand_b: The second operand

    Returns:
        {"success": True/False, "result": <number or None>, "error": <str or None>}
    """
    logger.info(f"Executing: {operand_a} {operation} {operand_b}")
    result = None
    error = None

    try:
        # TODO: Implement the operation logic
        # - "add":      operand_a + operand_b
        # - "subtract": operand_a - operand_b
        # - "multiply": operand_a * operand_b
        # - "divide":   operand_a / operand_b  (handle division by zero!)
        # - else:       set error = f"Unsupported operation: {operation}"
        pass
    except Exception as e:
        error = f"Calculation error: {str(e)}"

    if error:
        return {"success": False, "result": None, "error": error}
    return {"success": True, "result": result, "error": None}


# Quick test
print(execute_calculation("add", 10, 5))
print(execute_calculation("multiply", 500, 0.15))
print(execute_calculation("divide", 10, 0))

In [ ]:
# Validation check
from checker.lab01 import check_calculator_logic
check_calculator_logic(execute_calculation)

---
## Part 5: Wiring to the API — The Two-Call Pattern

LLM tool calling works as a two-step conversation:

```
1. You:  messages + tool schemas  →  LLM
   LLM: "Here are tool_calls I need you to execute"

2. You:  execute tools, append results  →  LLM
   LLM: "Based on the results, the answer is..."
```

### Walkthrough: `get_ai_response_with_tools`

In [ ]:
def get_ai_response_with_tools(
    messages: List[Dict[str, Any]],
    model: str = "openai/gpt-4o-mini"
) -> Dict[str, Any]:
    """
    Sends messages to the LLM via LiteLLM, handles tool calls, returns final text.
    Returns: {"response_text": str, "tool_results": list}
    """

    # --- First API Call: send messages + tool schemas ---
    try:
        response = litellm.completion(
            model=model,
            messages=messages,
            tools=[CALCULATOR_SCHEMA],
            tool_choice="auto",
            temperature=0.1
        )
    except Exception as e:
        logger.error(f"API call failed: {e}")
        return {"response_text": "Error connecting to API.", "tool_results": []}

    response_message = response.choices[0].message
    tool_results = []

    # --- Handle Tool Calls ---
    if response_message.get("tool_calls"):
        logger.info(f"Model initiated {len(response_message.tool_calls)} tool call(s).")

        # Append assistant message (with tool_calls) to history
        messages.append(response_message)

        # Execute each tool call
        for tool_call in response_message.tool_calls:
            tool_name = tool_call.function.name
            try:
                # Defensive parsing: arguments may arrive as a JSON string or as a dict
                args = tool_call.function.arguments
                if isinstance(args, str):
                    request = CalculationRequest.model_validate_json(args)
                else:
                    request = CalculationRequest.model_validate(args)

                tool_result = execute_calculation(**request.model_dump())
            except ValidationError as e:
                # In Pydantic v2, ValidationError captures both malformed JSON and schema mismatches
                tool_result = {"success": False, "error": f"Validation Error: {e}", "result": None}
            except Exception as e:
                tool_result = {"success": False, "error": f"Unexpected error: {e}", "result": None}

            tool_results.append(tool_result)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            })

        # --- Second API Call: get final answer ---
        second_response = litellm.completion(
            model=model, messages=messages, temperature=0.1
        )
        response_text = second_response.choices[0].message.content or "Calculation complete."
    else:
        response_text = response_message.content

    return {"response_text": response_text, "tool_results": tool_results}

---
## Part 6: Live Demo

Run a few questions through the agent and observe the tool being called.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant with access to a calculator. Use it for any math."

test_questions = [
    ("What is 15% of 200?", 30.0),
    ("If I multiply 1000 by 0.08, what do I get?", 80.0),
    ("What is 500 divided by 2?", 250.0),
]

for question, expected in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]
    result = get_ai_response_with_tools(messages)

    print(f"Q: {question}")
    print(f"A: {result['response_text']}")
    if result["tool_results"]:
        print(f"   [Tool called {len(result['tool_results'])} time(s)]")
        if str(expected) in result['response_text'] or str(int(expected)) in result['response_text']:
            print("   ✓ Correct answer found in response.")
        else:
            print("   ⚠ Expected answer not found in text. Check manually.")
    print()

---
## Part 7: Python Concept — Decorators

> [!NOTE]
> This section explains a Python pattern you'll use to build the registry below.
> If you're already comfortable with decorators, skim it and move on.

A **decorator** is a function that *wraps* another function to add behaviour without changing it.

```python
@some_decorator
def my_function():
    ...
```

This is exactly equivalent to:

```python
def my_function():
    ...
my_function = some_decorator(my_function)
```

The `@` syntax is just syntactic sugar. The decorator receives the function as an argument and returns a (usually modified) function.

**Decorators with arguments** add one more layer:

```python
@registry.register(name="my_tool", description="Does X")
def my_function():
    ...
```

Here `registry.register(name=..., description=...)` is called first, and it *returns* the actual decorator which is then applied to `my_function`.

Run the cell below to see this in action with a minimal example:

In [1]:
# ── Minimal decorator demo ─────────────────────────────────────────────────

tool_registry_demo = {}  # A simple dict to simulate a registry

def register_tool(name: str, description: str):
    """A decorator factory: takes config, returns a decorator."""
    def decorator(func):
        # Side effect: store the function in our registry
        tool_registry_demo[name] = {"func": func, "description": description}
        print(f"  Registered tool: '{name}'")
        return func   # Return the original function unchanged
    return decorator


# Now use it — Python calls register_tool("greet", "Says hello") first,
# which returns `decorator`, which is then applied to `say_hello`.
@register_tool(name="greet", description="Says hello to a user")
def say_hello(user: str):
    return f"Hello, {user}!"

@register_tool(name="farewell", description="Says goodbye to a user")
def say_goodbye(user: str):
    return f"Goodbye, {user}!"


print("\nTools registered:", list(tool_registry_demo.keys()))
print("Functions still work normally:", say_hello("Ali"))
print("Can call via registry:", tool_registry_demo["farewell"]["func"]("Sara"))

  Registered tool: 'greet'
  Registered tool: 'farewell'

Tools registered: ['greet', 'farewell']
Functions still work normally: Hello, Ali!
Can call via registry: Goodbye, Sara!


**Key insight**: The decorator:
1. Takes the function as input
2. Registers it as a side effect (stored in the registry)
3. Returns the function unchanged

This means you can write normal Python functions and just add `@registry.register(...)` to make them tools. No boilerplate classes needed.

---
## Part 8: Building a Plugin System — The Tool Registry

With one tool, a simple function works fine. But a real agent might have dozens of tools, and you need a way to:

- **Register** tools without modifying the agent loop
- **Get schemas** for all tools in one call
- **Dispatch** tool calls by name with validation
- **Rate-limit** expensive calls

This is the **plugin pattern** — the same pattern used in `project/src/tools/registry.py`.

We'll build it in **two pieces**. Read each one, then run it before moving on.

### Piece 1 of 2: `RegisteredTool` — wrap a function and auto-generate its schema

This class holds a single tool function. Its most important job is `_build_pydantic_model`:

> [!NOTE]
> **`_build_pydantic_model` uses advanced Python (metaprogramming).**  
> It reads your function's *type hints* at runtime using `inspect.signature`,
> then calls `create_model` to build a Pydantic class on-the-fly.
> The result: you never write a JSON schema by hand — just type-annotate your function
> and the schema is generated automatically.  
> You don't need to write this code — just understand what it achieves.

In [ ]:
class RegisteredTool:
    """Holds a function and its auto-generated schema."""

    def __init__(self, name: str, func: Callable, description: str):
        self.name = name
        self.func = func
        self.description = description
        self._model = self._build_pydantic_model(func)  # schema built once at registration

    def _build_pydantic_model(self, func: Callable) -> type[BaseModel]:
        """Reads the function's type hints and builds a Pydantic model automatically."""
        sig = inspect.signature(func)
        fields = {}
        for param_name, param in sig.parameters.items():
            annotation = param.annotation if param.annotation != inspect.Parameter.empty else str
            if param.default == inspect.Parameter.empty:
                fields[param_name] = (annotation, ...)           # required field
            else:
                fields[param_name] = (annotation, param.default) # optional with default
        return create_model(f"{self.name}Input", **fields)

    def get_schema(self) -> Dict[str, Any]:
        """Returns the OpenAI-compatible function schema."""
        raw = self._model.model_json_schema()
        raw.pop("title", None)  # strip Pydantic metadata the API doesn't need
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": raw,
            }
        }

    def execute(self, **kwargs) -> Dict[str, Any]:
        """Validates arguments with Pydantic, then calls the function."""
        # Pydantic v2 recommendation: use model_validate for dict inputs
        validated = self._model.model_validate(kwargs)
        return self.func(**validated.model_dump())


print("RegisteredTool ready.")

### Piece 2 of 2: `ToolRegistry` — the plugin manager

This is the class you'll interact with every day. It has three responsibilities:

| Method | What it does |
|--------|--------------|
| `register(name, description)` | Decorator factory — wraps your function in a `RegisteredTool` |
| `get_schemas()` | Returns all tool schemas to pass to the LLM API |
| `execute(tool_name, arguments)` | Dispatches a tool call by name with validation |

In [ ]:
class ToolRegistry:
    """Decorator-based plugin registry for LLM tools."""

    def __init__(self):
        self._tools: Dict[str, RegisteredTool] = {}

    def register(self, name: str, description: str):
        """
        Decorator factory: registers the decorated function as a tool.

        Usage:
            @registry.register(name="my_tool", description="Does X")
            def my_tool(arg1: str, arg2: int) -> Dict[str, Any]:
                ...
        """
        def decorator(func: Callable):
            tool = RegisteredTool(name, func, description)
            self._tools[name] = tool
            logger.info(f"Registered tool: '{name}'")
            return func  # return the original function unchanged
        return decorator

    def get_schemas(self) -> List[Dict[str, Any]]:
        """Returns all tool schemas for the LLM API call."""
        return [tool.get_schema() for tool in self._tools.values()]

    def execute(self, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """Executes a tool by name."""
        tool = self._tools.get(tool_name)
        if not tool:
            return {"success": False, "result": None, "error": f"Tool '{tool_name}' not found."}
        try:
            return tool.execute(**arguments)
        except Exception as e:
            logger.error(f"Execution failed for {tool_name}: {e}", exc_info=True)
            return {"success": False, "result": None, "error": f"Internal error: {str(e)}"}


# Create the shared registry instance used for the rest of this lab
registry = ToolRegistry()
print("ToolRegistry ready.")

### Step 2: Register the Calculator Tool — Walkthrough

Now we register the `execute_calculation` function from Part 3.
Notice how the **function body does not change** — only the `@registry.register(...)` decorator is added.
The registry reads the type hints, builds the schema, and stores everything automatically.

In [ ]:
@registry.register(
    name="execute_calculation",
    description=(
        "Executes a basic arithmetic operation. "
        "Use for any math: percentages, growth rates, compound interest, or simple arithmetic. "
        "Example: 15% of 200 → operation='multiply', operand_a=200, operand_b=0.15."
    ),
)
def execute_calculation_v2(operation: str, operand_a: float, operand_b: float) -> Dict[str, Any]:
    """Registered version of the calculator."""
    try:
        if operation == "add":
            result = operand_a + operand_b
        elif operation == "subtract":
            result = operand_a - operand_b
        elif operation == "multiply":
            result = operand_a * operand_b
        elif operation == "divide":
            if operand_b == 0:
                return {"success": False, "result": None, "error": "Division by zero is not allowed."}
            result = operand_a / operand_b
        else:
            return {"success": False, "result": None, "error": f"Unsupported operation: {operation}"}
        return {"success": True, "result": result, "error": None}
    except Exception as e:
        return {"success": False, "result": None, "error": f"Calculation error: {str(e)}"}


# Inspect what was registered
print("Registered tools:", list(registry._tools.keys()))
print("\nAuto-generated schema:")
print(json.dumps(registry.get_schemas()[0], indent=2))

**Quick check** — answer these mentally before moving to the exercise:

1. What does `registry.get_schemas()` return, and where is it used?
2. If you add `@registry.register(name='foo', ...)` to a new function, which method stores it?
3. What happens if `execute` is called with `tool_name='nonexistent_tool'`?

> *Answers: (1) a list of OpenAI-compatible dicts, passed to `litellm.completion(tools=...)`;*  
> *(2) `register`'s inner `decorator` stores it in `self._tools`;*  
> *(3) `execute` returns `{"success": False, "error": "Tool 'nonexistent_tool' not found."}` — never raises.*

---

### Exercise: Add a `get_exchange_rate` Tool

Your turn. Register a new tool that simulates a currency exchange rate lookup.

**Requirements:**
- Name: `get_exchange_rate`
- Parameters: `from_currency: str`, `to_currency: str`
- Returns the structured result dict (success/result/error contract)
- For this exercise, **simulate** rates with a fixed dict — no real API needed
- Register it on the **same `registry` instance** so the agent can use it automatically

**Use fake rates:**
```python
RATES = {
    ("USD", "SAR"): 3.75,
    ("EUR", "SAR"): 4.10,
    ("USD", "EUR"): 0.92,
}
```

In [ ]:
RATES = {
    ("USD", "SAR"): 3.75,
    ("EUR", "SAR"): 4.10,
    ("USD", "EUR"): 0.92,
}

# TODO: Write the @registry.register(...) decorator with a good name + description
# TODO: Define the function signature with correct type hints
# TODO: Look up the rate in RATES and return the structured result
# Hint: try RATES.get((from_currency.upper(), to_currency.upper()))

# After completing, run this to verify:
print("Tools registered:", list(registry._tools.keys()))
print("\nTest call:")
print(registry.execute("get_exchange_rate", {"from_currency": "USD", "to_currency": "SAR"}))
print(registry.execute("get_exchange_rate", {"from_currency": "GBP", "to_currency": "SAR"}))

---
## Part 9: Live Demo with the Registry

Now we update the agent loop to use `registry.get_schemas()` and `registry.execute()` instead of hardcoded calls.
This is the exact pattern in `project/src/tools/registry.py` — adding a new tool only requires a new `@registry.register(...)` function. The agent loop never changes.

### Walkthrough: Registry-aware agent loop

In [ ]:
def run_agent(user_message: str, model: str = "openai/gpt-4o-mini") -> str:
    """
    A minimal agent loop backed by the ToolRegistry.
    Adding a new tool only requires @registry.register(...) — nothing here changes.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with access to tools. "
                "Use the calculator for math and exchange rates for currency conversions."
            )
        },
        {"role": "user", "content": user_message}
    ]

    # --- First call: pass ALL registered tool schemas ---
    try:
        response = litellm.completion(
            model=model,
            messages=messages,
            tools=registry.get_schemas(),   # <-- registry provides all schemas
            tool_choice="auto",
            temperature=0.1
        )
    except Exception as e:
        return f"API error: {e}"

    response_message = response.choices[0].message

    if response_message.get("tool_calls"):
        messages.append(response_message)

        for tool_call in response_message.tool_calls:
            tool_name = tool_call.function.name
            try:
                # Defensive parsing: arguments may arrive as a JSON string or as a dict
                args = tool_call.function.arguments
                arguments = json.loads(args) if isinstance(args, str) else args
                result = registry.execute(tool_name, arguments)  # <-- registry dispatches
            except json.JSONDecodeError:
                result = {"success": False, "result": None, "error": "Invalid JSON arguments"}
            except Exception as e:
                result = {"success": False, "result": None, "error": str(e)}

            logger.info(f"Tool '{tool_name}' result: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

        final = litellm.completion(model=model, messages=messages, temperature=0.1)
        return final.choices[0].message.content or "Done."

    return response_message.content


# Run a mixed workload — both tools get used
questions = [
    "What is 15% of 200?",
    "What is the exchange rate from USD to SAR?",
    "I have $500. How much is that in SAR?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {run_agent(q)}")
    print()

---
## Part 10: Parallel Tool Calls — Measuring `asyncio.gather`

Section G of the slides argued that independent tool calls don't need to wait in line. Let's measure it.

We'll define two **fake-latency tools** (each sleeps 1 second) and run them two ways:

1. **Sequentially** — one `await` after another.
2. **In parallel** — both wrapped in a single `asyncio.gather`.

The two tools have no data dependency between them, so they're a fair parallelisation candidate. If `lookup_b` needed `lookup_a`'s result first, you'd have to keep them sequential — the bottleneck is correctness, not the framework.

In [ ]:
import asyncio
import time

# Two independent fake-latency tools. Each "API call" takes 1 second.
async def lookup_a():
    await asyncio.sleep(1.0)
    return {"success": True, "result": "result A", "error": None}

async def lookup_b():
    await asyncio.sleep(1.0)
    return {"success": True, "result": "result B", "error": None}


# --- Sequential: await one, then the next ---
async def run_sequential():
    t0 = time.perf_counter()
    a = await lookup_a()
    b = await lookup_b()
    return time.perf_counter() - t0, [a, b]


# --- Parallel: dispatch both, then await both ---
async def run_parallel():
    t0 = time.perf_counter()
    a, b = await asyncio.gather(lookup_a(), lookup_b())
    return time.perf_counter() - t0, [a, b]


# Jupyter has a running event loop, so use top-level await
seq_elapsed, _ = await run_sequential()
par_elapsed, _ = await run_parallel()

print(f"Sequential: {seq_elapsed:.2f}s   (expected ~2.0s)")
print(f"Parallel:   {par_elapsed:.2f}s   (expected ~1.0s)")
print(f"Speedup:    {seq_elapsed / par_elapsed:.2f}x")

### Exercise: When does parallelism *not* help?

1. Add a third tool `lookup_c()` (same fake-latency pattern). Run all three in parallel — what speedup do you get?
2. Now imagine `lookup_c` needs `lookup_a`'s result before it can run. Why can't you use a single `asyncio.gather` for all three? Sketch the dependency: which calls *can* still be parallelised, and which one has to wait?

> **The rule from the slides:** parallelise calls that share no **data dependencies**. The framework will gladly let you parallelise dependent calls — but you'll get a wrong answer (or worse, a race) very fast.

---
## Reflection

### Key Takeaways

| Concept | What you learned |
|---------|------------------|
| **Schema description** | The LLM uses the description to decide *when* to call a tool — treat it as a prompt |
| **Enums** | Dramatically improve accuracy by constraining the model's output space |
| **Pydantic** | Generates schemas and validates LLM output in one step |
| **Structured errors** | Never let a tool raise an exception into the agent loop |
| **Two-call pattern** | Tool use is a conversation turn, not a function call |
| **Decorators** | `@registry.register(...)` is syntactic sugar for `func = registry.register(...)(func)` |
| **Plugin registry** | Decouple tools from the agent loop — add tools without touching agent code |

### Connection to the Project

Open `project/src/tools/registry.py`. You'll recognise everything:
- `Tool` wraps a function and auto-generates its schema (like `RegisteredTool` here)
- `ToolRegistry.register(...)` is the same decorator pattern you built
- `registry.get_all_tools()` feeds schemas to the agent loop

The project extends this with categories and stricter schema validation — but the core idea is identical to what you built today.

### Bonus Challenge

Register a third tool — `get_weather(city: str, unit: str)` — and add it to `run_agent`.
Simulate weather data with a dict. Observe that the agent loop in Part 9 requires **zero changes**.